In [23]:
from typing import Optional
from datetime import datetime
import pandas as pd
import numpy as np
import nfl_data_py as nfl
import os
from pathlib import Path

from sqlalchemy import (
    create_engine, Column, Integer, Float, String, Boolean,
    Date, DateTime, Text, ForeignKey, Index, UniqueConstraint
)
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from sqlalchemy.pool import StaticPool

pd.set_option('display.max_columns', None)

import cfbd
import nfl_data_py as nfl

CFBD_API_KEY = "xOR6a8qPGapvDyx02MO5wgJPY6fkf7EMT40z030ZwRz/DxG9hB9RF+lc1zyJ0n6U"

POWER_5 = {"SEC", "Big Ten", "Big 12", "ACC", "Pac-12", "Pac-10"}
GROUP_OF_5 = {"American Athletic", "Mountain West", "Conference USA", "MAC", "Sun Belt"}

# Positions to track for fantasy relevance
COLLEGE_POSITIONS = {"QB", "RB", "WR", "TE"}
STAT_CATEGORIES = ["passing", "rushing", "receiving"]

In [24]:
config = cfbd.Configuration(access_token=CFBD_API_KEY)
config.api_key["Authorization"] = CFBD_API_KEY
config.api_key_prefix["Authorization"] = "Bearer"
cfbd_client = cfbd.ApiClient(config)
stats_api = cfbd.StatsApi(cfbd_client)
players_api = cfbd.PlayersApi(cfbd_client)
teams_api = cfbd.TeamsApi(cfbd_client)
games_api = cfbd.GamesApi(cfbd_client)

In [238]:
player_stats = stats_api.get_player_season_stats(
                year=year, category="passing"
            )
player_stats

[PlayerStat(season=2014, player_id='381494', player='Christian Stewart', position='QB', team='BYU', conference='FBS Independents', category='passing', stat_type='ATT', stat='348'),
 PlayerStat(season=2014, player_id='381494', player='Christian Stewart', position='QB', team='BYU', conference='FBS Independents', category='passing', stat_type='COMPLETIONS', stat='199'),
 PlayerStat(season=2014, player_id='381494', player='Christian Stewart', position='QB', team='BYU', conference='FBS Independents', category='passing', stat_type='INT', stat='9'),
 PlayerStat(season=2014, player_id='381494', player='Christian Stewart', position='QB', team='BYU', conference='FBS Independents', category='passing', stat_type='PCT', stat='0.572'),
 PlayerStat(season=2014, player_id='381494', player='Christian Stewart', position='QB', team='BYU', conference='FBS Independents', category='passing', stat_type='TD', stat='25'),
 PlayerStat(season=2014, player_id='381494', player='Christian Stewart', position='QB', t

In [234]:
rosters = nfl.import_players()
rosters

,gsis_id,display_name,common_first_name,first_name,last_name,short_name,football_name,suffix,esb_id,nfl_id,pfr_id,pff_id,otc_id,espn_id,smart_id,birth_date,position_group,position,ngs_position_group,ngs_position,height,weight,headshot,college_name,college_conference,jersey_number,rookie_season,last_season,latest_team,status,ngs_status,ngs_status_short_description,years_of_experience,pff_position,pff_status,draft_year,draft_round,draft_pick,draft_team
0,00-0028830,Isaako Aaitui,Isaako,Isaako,Aaitui,None,None,None,AAI622937,None,AaitIs00,6998,2535,14856,32004141-4962-2937-61ff-017b1804dec6,1987-01-25,DL,NT,None,None,76.0,307.0,https://static.www.nfl.com/image/private/f_aut...,UNLV,None,0,2011,2014,WAS,DEV,None,None,2,DI,None,NaN,NaN,NaN,None
1,00-0038389,Israel Abanikanda,Israel,Israel,Abanikanda,I.Abanikanda,Israel,None,ABA159567,56008,AbanIs00,122999,10967,4429202,32004142-4115-9567-2e24-0eab29f6a4b9,2002-10-05,RB,RB,RB,RB,70.0,216.0,https://static.www.nfl.com/image/upload/f_auto...,Pittsburgh,Atlantic Coast Conference,30,2023,2026,DAL,ACT,RES,Reserve/Future,3,HB,A,2023.0,5.0,143.0,NYJ
2,00-0024644,Jon Abbate,Jon,Jon,Abbate,None,None,None,ABB051371,None,None,None,None,None,32004142-4205-1371-db95-1abc96313b69,1985-06-18,LB,LB,None,None,71.0,245.0,https://static.www.nfl.com/image/private/f_aut...,Wake Forest,None,67,2007,2007,HOU,RES,None,None,0,None,None,NaN,NaN,NaN,None
3,ABB498348,Vince Abbott,Vince,Vincent,Abbott,None,None,None,ABB498348,None,abbotvin01,None,None,None,32004142-4249-8348-e00f-5fbbe6a0c73c,1958-05-31,SPEC,K,None,None,71.0,207.0,https://static.www.nfl.com/image/private/f_aut...,California State-Fullerton; Washington,None,0,1987,1988,LAC,ACT,None,None,2,None,None,NaN,NaN,NaN,None
4,00-0031021,Jared Abbrederis,Jared,Jared,Abbrederis,J.Abbrederis,Jared,None,ABB650964,41405,AbbrJa00,8811,3115,16836,32004142-4265-0964-fc36-bb0ad76ff6e6,1990-12-17,WR,WR,WR,WR,73.0,195.0,https://static.www.nfl.com/image/private/f_aut...,Wisconsin,None,10,2014,2017,DET,CUT,CUT,None,4,WR,None,2014.0,5.0,176.0,GB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24961,DEB622935,Case deBruijn,Case,Case,deBruijn,None,None,None,DEB622935,None,deBrCa20,None,None,None,32004445-4262-2935-c44d-416ca8a4c116,1960-04-11,SPEC,P,None,None,72.0,176.0,https://static.www.nfl.com/image/private/f_aut...,Idaho State,None,0,1982,1982,KC,ACT,None,None,0,None,None,1982.0,8.0,214.0,KC
24962,VAN516304,Mark van Eeghen,Mark,Mark,van Eeghen,None,None,None,VAN516304,None,VanEMa00,None,None,None,32005641-4e51-6304-9ea7-e13b6d636311,1952-04-19,RB,RB,None,None,74.0,223.0,https://static.www.nfl.com/image/private/f_aut...,Colgate,None,0,1974,1983,NE,ACT,None,None,10,None,None,1974.0,3.0,75.0,LV
24963,VAN366267,Jordan van den Berg,Jordan,Jordan,van den Berg,None,None,None,VAN366267,None,vandJo00,None,None,4875381,32005641-4e36-6267-1a50-c0be451e3b91,2002-04-12,DL,DT,None,None,75.0,310.0,None,Georgia Tech; Penn State; Iowa Western CC,None,None,2026,2026,CHI,ACT,None,None,0,None,None,NaN,NaN,NaN,None
24964,00-0016956,Kimo von Oelhoffen,Kimo,Kimo,von Oelhoffen,None,None,None,VON221488,None,vonOKi20,None,None,565,3200564f-4e22-1488-f980-8c2faa904183,1971-01-30,DL,DT,None,None,76.0,299.0,https://static.www.nfl.com/image/private/f_aut...,Boise State; Hawaii; Walla Walla Community Col...,None,66,1994,2007,PHI,ACT,None,None,14,None,None,1994.0,6.0,162.0,CIN


In [25]:
current_path = os.getcwd()
DB_PATH = str(Path(current_path).parent / "dynasty_scout.db")
DB_PATH

'/Users/sandeeptiwari/Desktop/dynasty-ai-engine/notebooks/dynasty_scout.db'

In [26]:
def get_engine(db_path: Path = DB_PATH, echo: bool = False):
    """
    Returns a SQLAlchemy engine. Uses StaticPool so the same connection
    is reused in single-threaded contexts (fine for local use).
    """
    return create_engine(
        f"sqlite:///{db_path}",
        connect_args={"check_same_thread": False},
        poolclass=StaticPool,
        echo=echo,
    )

def get_session(engine):
    return Session(engine)

In [27]:
engine = get_engine()

In [28]:
def _ingest_year(year: int):
    """
    Pull all player stats and team stats for one year, compute context, store
    """
    # ---- Team stats (needed for dominator rating denominator) ----
    raw_team_stats = stats_api.get_team_stats(year=year)
    team_context = _build_team_context(raw_team_stats)

    # ---- Player stats ----
    # CFBD returns stats by category. We pull each one separately.
    stats = []
    for category in STAT_CATEGORIES:
        try:
            player_stats = stats_api.get_player_season_stats(
                year=year, category=category
            )
            for stat in player_stats:
                stats.append((category, stat))
        except Exception as e:
            logger.warning(f"{category} stats for {year} failed: {e}")
        
    # Group by (player, team) and merge all categories
    player_data: dict = {}  # key: (player_id, team)
    for category, stat in stats:
        key = (stat.player_id, stat.team)
        if key not in player_data:
            player_data[key] = {
                "cfbd_player_id": stat.player_id,
                "player_name": stat.player,
                "team": stat.team,
                "season": year,
                "conference": None,
                "position": None,
            }
        print(player_data)
        # _merge_stat_row(player_data[key], category, stat)
        break
    
    # Enrich with conference and position from player search
    _enrich_player_metadata(player_data, year)

    # Build ORM records and bulk upsert
    records_created = 0
    # with get_session(self.engine) as session:
    #     for key, data in player_data.items():
    #         team = data.get("team")
    #         team_ctx = team_context.get(team, {})
    #         record = self._build_college_stats_record(data, team_ctx)
    #         if record:
    #             session.merge(record)
    #             records_created += 1
    #     session.commit()
    # logger.info(f"  → {records_created} college season records ingested for {year}")

def _build_team_context(raw_team_stats) -> dict:
    """
    Build team stat totals keyed by team name for dominator rating
    """
    context: dict = {}
    for team_info in raw_team_stats:
        team = team_info.team
        if team not in context:
            context[team] = {}
        stat_name = team_info.stat_name
        print("STAT NAME", stat_name)
        # Accumulate relevant team totals
        if stat_name == "passingYards":
            context[team]["team_pass_yards"] = team_info.stat_value
        elif stat_name == "passingTDs":
            context[team]["team_pass_tds"] = team_info.stat_value
        elif stat_name == "receivingYards":
            context[team]["team_rec_yards"] = team_info.stat_value
        elif stat_name == "receivingTDs":
            context[team]["team_rec_tds"] = team_info.stat_value
    return context
    
def _merge_stat_row(data: dict, category: str, row) -> None:
    """
    Merge a single stat category row into the player's data dict
    """
    if category == "passing":
        data.update({
            "pass_completions": getattr(row, "completions", None),
            "pass_attempts": getattr(row, "att", None),
            "pass_yards": getattr(row, "yds", None),
            "pass_tds": getattr(row, "td", None),
            "interceptions": getattr(row, "int", None),
        })
    elif category == "rushing":
        data.update({
            "rush_carries": getattr(row, "car", None),
            "rush_yards": getattr(row, "yds", None),
            "rush_tds": getattr(row, "td", None),
        })
    elif category == "receiving":
        data.update({
            "receptions": getattr(row, "rec", None),
            "rec_yards": getattr(row, "yds", None),
            "rec_tds": getattr(row, "td", None),
        })

def _enrich_player_metadata(self, player_data: dict, year: int):
    """
    Call CFBD player search to get position and conference.
    Batches by team to minimize API calls
    """
    teams = set(team for (_, team) in player_data)
    team_rosters: dict = {}
    for team in teams:
        try:
            roster = self.teams_api.get_roster(team=team, year=year)
            for player in roster:
                team_rosters[(player.id, team)] = {
                    "position": player.position,
                    "conference": getattr(player, "conference", None),
                    "games": getattr(player, "games", None),
                }
            time.sleep(0.1)
        except Exception as e:
            logger.warning(e)
            pass

    for key, data in player_data.items():
        enrichment = team_rosters.get(key, {})
        data["position"] = enrichment.get("position") or data.get("position")
        data["conference"] = enrichment.get("conference") or data.get("conference")
        data["games_played"] = enrichment.get("games") or data.get("games_played")

### Team Stats

In [29]:
year = 2020

In [30]:
raw_team_stats = stats_api.get_team_stats(year=year)
# team_context = _build_team_context(raw_team_stats)

In [31]:
print(set([t.stat_name for t in raw_team_stats]))

{'kickReturnYards', 'sacks', 'interceptionYardsOpponent', 'passesInterceptedOpponent', 'rushingTDsOpponent', 'fourthDownConversionsOpponent', 'firstDowns', 'fumblesRecovered', 'possessionTime', 'fumblesLost', 'puntReturnYards', 'rushingTDs', 'passingTDsOpponent', 'fourthDownsOpponent', 'netPassingYards', 'kickReturnYardsOpponent', 'thirdDownConversions', 'interceptionsOpponent', 'kickReturnTDs', 'fumblesRecoveredOpponent', 'totalYards', 'totalYardsOpponent', 'games', 'passesIntercepted', 'passCompletions', 'sacksOpponent', 'passAttempts', 'passAttemptsOpponent', 'puntReturnTDs', 'kickReturns', 'penaltyYards', 'thirdDowns', 'interceptions', 'tacklesForLoss', 'netPassingYardsOpponent', 'interceptionTDs', 'puntReturnTDsOpponent', 'passCompletionsOpponent', 'fourthDowns', 'possessionTimeOpponent', 'turnoversOpponent', 'passingTDs', 'rushingAttempts', 'puntReturns', 'puntReturnYardsOpponent', 'tacklesForLossOpponent', 'penaltiesOpponent', 'interceptionYards', 'kickReturnTDsOpponent', 'fumbl

In [32]:
x = {'kickReturnYards', 'sacks', 'interceptionYardsOpponent', 'passesInterceptedOpponent', 'rushingTDsOpponent', 'fourthDownConversionsOpponent', 'firstDowns', 'fumblesRecovered', 'possessionTime', 'fumblesLost', 'puntReturnYards', 'rushingTDs', 'passingTDsOpponent', 'fourthDownsOpponent', 'netPassingYards', 'kickReturnYardsOpponent', 'thirdDownConversions', 'interceptionsOpponent', 'kickReturnTDs', 'fumblesRecoveredOpponent', 'totalYards', 'totalYardsOpponent', 'games', 'passesIntercepted', 'passCompletions', 'sacksOpponent', 'passAttempts', 'passAttemptsOpponent', 'puntReturnTDs', 'kickReturns', 'penaltyYards', 'thirdDowns', 'interceptions', 'tacklesForLoss', 'netPassingYardsOpponent', 'interceptionTDs', 'puntReturnTDsOpponent', 'passCompletionsOpponent', 'fourthDowns', 'possessionTimeOpponent', 'turnoversOpponent', 'passingTDs', 'rushingAttempts', 'puntReturns', 'puntReturnYardsOpponent', 'tacklesForLossOpponent', 'penaltiesOpponent', 'interceptionYards', 'kickReturnTDsOpponent', 'fumblesLostOpponent', 'rushingYards', 'rushingYardsOpponent', 'penaltyYardsOpponent', 'turnovers', 'thirdDownsOpponent', 'interceptionTDsOpponent', 'thirdDownConversionsOpponent', 'fourthDownConversions', 'penalties', 'kickReturnsOpponent', 'puntReturnsOpponent', 'rushingAttemptsOpponent', 'firstDownsOpponent'}




In [33]:
[c for c in x if 'rec' in c.lower()]

['fumblesRecovered', 'fumblesRecoveredOpponent']

In [34]:
category = "passing"
player_stats = stats_api.get_player_season_stats(
                    year=year, category=category
                )

In [35]:
stats_dict = {}

for category in STAT_CATEGORIES:
    try:
        player_stats = stats_api.get_player_season_stats(
            year=year, category=category
        )
        stats_dict[category] = player_stats
        # for stat in player_stats:
        #     stats.append((category, stat))
    except Exception as e:
        print(f"{category} stats for {year} failed: {e}")

In [36]:
PLAYER_STATS = {'passing': {
    'ATT': 'pass_attempts',
    'COMPLETIONS': 'pass_completions',
    'INT': 'interceptions',
    'PCT': 'completion_pct',
    'TD': 'pass_tds',
    'YDS': 'pass_yds',
    'YPA': 'yards_per_attempt'
    },
 'rushing': {
     'CAR': 'rush_attempts',
     'LONG': 'longest_rush_attempt',
     'TD': 'rush_tds',
     'YDS': 'rush_yds',
     'YPC': 'yards_per_carry'
 },
 'receiving': {
     'LONG': 'longest_reception',
     'REC': 'receptions',
     'TD': 'receiving_tds',
     'YDS': 'receiving_yds',
     'YPR': 'yards_per_reception',
 }
}

def _merge_stat_row(data: dict, category: str, row) -> None:
    """
    Merge a single stat category row into the player's data dict
    """
    stat_types = PLAYER_STATS[category]
    stat_type = row.stat_type
    stat_val = row.stat

    data.update(
        {stat_types[stat_type]: float(stat_val) if '.' in stat_val else int(stat_val)}
    )

In [38]:
i = 0

player_data = {}
for category, player_stats_lst in stats_dict.items():
    print(category)
    for player_stat in player_stats_lst:
        # print(player_stat)
        key = (player_stat.player_id, player_stat.team)
        if key not in player_data:
            player_data[key] = {
                "cfbd_player_id": player_stat.player_id,
                "player_name": player_stat.player,
                "team": player_stat.team,
                "season": year,
                "conference": player_stat.conference,
                "position": player_stat.position,
            }
        _merge_stat_row(player_data[key], category, player_stat)
        

passing
rushing
receiving


In [82]:
teams = set(team for (_, team) in player_data)
team_rosters: dict = {}
for team in teams:
    try:
        roster = teams_api.get_roster(team=team, year=year)
        for player in roster:
            team_rosters[(player.id, team)] = {
                "position": player.position,
                "conference": player_data[(player.id, team)]['conference'],
                "games": getattr(player, "games", None),
            }
        time.sleep(0.1)
    except Exception as e:
        print(e)
    print(team_rosters)
    break

('4264584', 'Western Carolina')
{('4032487', 'Western Carolina'): {'position': 'RB', 'conference': 'Southern', 'games': None}, ('4032504', 'Western Carolina'): {'position': 'WR', 'conference': 'Southern', 'games': None}, ('4250525', 'Western Carolina'): {'position': 'QB', 'conference': 'Southern', 'games': None}, ('4250526', 'Western Carolina'): {'position': 'DB', 'conference': 'Southern', 'games': None}, ('4250545', 'Western Carolina'): {'position': 'TE', 'conference': 'Southern', 'games': None}, ('4250552', 'Western Carolina'): {'position': 'RB', 'conference': 'Southern', 'games': None}}


In [186]:
for key, data in player_data.items():
    print(data["player_id"])

KeyError: 'player_id'

In [41]:
# i = 0
# stats = set()
def _unwrap_stat(val):
    # If it's the cfbd model wrapper, get actual_instance; otherwise use value directly
    try:
        inner = getattr(val, "actual_instance", val)
        return float(inner) if inner is not None else 0.0
    except Exception:
        try:
            return float(val)
        except Exception:
            return 0.0
                
context = {}
for team_info in raw_team_stats:
    team = team_info.team
    if team not in context:
        context[team] = {}
    stat_name = team_info.stat_name
    # print(stat_name)
    if stat_name == "netPassingYards":
        context[team]["team_pass_yards"] = _unwrap_stat(team_info.stat_value)
        context[team]["team_rec_yards"] = _unwrap_stat(team_info.stat_value)
    elif stat_name == "passingTDs":
        context[team]["team_pass_tds"] = _unwrap_stat(team_info.stat_value)
        context[team]["team_rec_tds"] = _unwrap_stat(team_info.stat_value)
    # elif stat_name == "receivingYards":
    #     context[team]["team_rec_yards"] = _unwrap_stat(team_info.stat_value)
    # elif stat_name == "receivingTDs":
    #     context[team]["team_rec_tds"] = _unwrap_stat(team_info.stat_value)

In [42]:
def _build_college_stats_record(data, team_ctx):
    if not data.get("cfbd_player_id"):
        return None

    receptions = data.get("receptions", 0) or 0
    rec_yards = data.get("receiving_yds", 0) or 0
    rec_tds = data.get("receiving_tds", 0) or 0

    rush_yards = data.get("rush_yards", 0) or 0
    rush_carries = data.get("rush_carries", 1) or 1
    rush_tds = data.get("rush_tds", 0) or 0

    # Coerce team stat wrapper types (e.g. TeamStatStatValue) to numeric safely
    def _unwrap_stat(val):
        # If it's the cfbd model wrapper, get actual_instance; otherwise use value directly
        try:
            inner = getattr(val, "actual_instance", val)
            return float(inner) if inner is not None else 0.0
        except Exception:
            try:
                return float(val)
            except Exception:
                return 0.0

    # Team totals (CFBD provides passing and receiving totals; rushing may not be present)
    team_pass_y = team_ctx.get("team_pass_yards")
    team_pass_t = team_ctx.get("team_pass_tds")
    team_rec_y = team_ctx.get("team_rec_yards")
    team_rec_t = team_ctx.get("team_rec_tds")

    # print(team_pass_y, team_pass_t, team_rec_y, team_rec_t)

#### Games Played

In [352]:
def calc_player_games_played_by_team(team_name, season):
    game_player_stats = games_api.get_game_player_stats(
        year=season,
        team=team_name
    )
    player_games_appeared = {}
    
    for game in game_player_stats:
        teams = game.to_dict()['teams']
        team_stats = [team for team in teams if team['team'] == team_name][0]
        categories = team_stats['categories']
        players_seen = set()
        for category in categories:
            cat_types = category['types']
            for cat_type in cat_types:
                athletes = cat_type['athletes']
                for athlete in athletes:
                    players_seen.add(athlete['id'])
    
        for player_seen in players_seen:
            player_games_appeared[player_seen] = player_games_appeared.get(player_seen, 0) + 1
    return player_games_appeared

In [347]:
team_name = 'California'
year = 2014

# calc_player_games_played_by_team(team_name, year)
game_player_stats = games_api.get_game_player_stats(
        year=year,
        team=team_name,
        
    )
game_player_stats

[GamePlayerStats(id=400547975, teams=[GamePlayerStatsTeam(team='California', conference='Pac-12', home_away='away', points=31, categories=[GamePlayerStatCategories(name='passing', types=[GamePlayerStatTypes(name='C/ATT', athletes=[GamePlayerStatPlayer(id='547401', name='Jared Goff', stat='21/34'), GamePlayerStatPlayer(id='3127202', name='Luke Rubenzer', stat='2/5')]), GamePlayerStatTypes(name='YDS', athletes=[GamePlayerStatPlayer(id='547401', name='Jared Goff', stat='281'), GamePlayerStatPlayer(id='3127202', name='Luke Rubenzer', stat='19')]), GamePlayerStatTypes(name='AVG', athletes=[GamePlayerStatPlayer(id='547401', name='Jared Goff', stat='8.3'), GamePlayerStatPlayer(id='3127202', name='Luke Rubenzer', stat='3.8')]), GamePlayerStatTypes(name='TD', athletes=[GamePlayerStatPlayer(id='547401', name='Jared Goff', stat='3'), GamePlayerStatPlayer(id='3127202', name='Luke Rubenzer', stat='0')]), GamePlayerStatTypes(name='INT', athletes=[GamePlayerStatPlayer(id='547401', name='Jared Goff', 

In [348]:
player_games_appeared = {}

for game in game_player_stats:
    teams = game.teams
    # print(len(teams))
    for team in teams:
        categories = team.categories
        players_seen = set()
        for category in categories:
            types = category.types
            for cat_type in types:
                athletes = cat_type.athletes
                # print(athletes)
                for athlete in athletes:
                    players_seen.add(athlete.id)
        for player_seen in players_seen:
            player_games_appeared[player_seen] = player_games_appeared.get(player_seen, 0) + 1
player_games_appeared

{'535284': 3,
 '550296': 10,
 '3127202': 10,
 '517016': 12,
 '3127207': 8,
 '550289': 7,
 '517011': 9,
 '535178': 9,
 '535269': 12,
 '535181': 12,
 '535271': 12,
 '516956': 9,
 '535270': 11,
 '535265': 12,
 '547401': 12,
 '3127200': 7,
 '517015': 1,
 '518189': 1,
 '500405': 1,
 '500400': 1,
 '518315': 1,
 '518316': 1,
 '518343': 1,
 '3116145': 1,
 '500412': 1,
 '533334': 1,
 '3116136': 1,
 '518311': 1,
 '533308': 1,
 '500254': 1,
 '546829': 1,
 '533321': 1,
 '546836': 1,
 '500419': 1,
 '3127211': 1,
 '517022': 1,
 '504731': 1,
 '517708': 1,
 '546925': 1,
 '546920': 1,
 '498266': 1,
 '537912': 1,
 '546921': 1,
 '529471': 1,
 '517728': 1,
 '517736': 1,
 '512633': 1,
 '537915': 1,
 '529469': 1,
 '517735': 1,
 '498236': 1,
 '3120946': 1,
 '535263': 1,
 '3140773': 1,
 '516997': 10,
 '500475': 1,
 '550667': 1,
 '550267': 1,
 '513736': 1,
 '550278': 1,
 '550277': 1,
 '550285': 1,
 '3122607': 1,
 '511429': 1,
 '511433': 1,
 '550669': 1,
 '3122620': 1,
 '535778': 1,
 '535774': 1,
 '500465': 1,


In [146]:
roster = teams_api.get_roster(team=team_name, year=year)

for player in roster:
    if getattr(player, "games", None):
        print(player)

In [44]:
for key, data in player_data.items():
    team = data.get("team")
    team_ctx = context.get(team, {})
    print(data)
    record = _build_college_stats_record(data, team_ctx)
    print(record)
    break

{'cfbd_player_id': '3858269', 'player_name': 'Ross Bowers', 'team': 'Northern Illinois', 'season': 2020, 'conference': 'Mid-American', 'position': 'QB', 'pass_attempts': 212, 'pass_completions': 123, 'interceptions': 2, 'completion_pct': 0.58, 'pass_tds': 10, 'pass_yds': 1365, 'yards_per_attempt': 6.4, 'rush_attempts': 21, 'longest_rush_attempt': 7, 'rush_tds': 0, 'rush_yds': -87, 'yards_per_carry': -4.1}
None


In [138]:
{k: v for k,v in player_data.items() if v['position'] == 'WR'}

{('4032247', 'The Citadel'): {'cfbd_player_id': '4032247',
  'player_name': 'Raleigh Webb',
  'team': 'The Citadel',
  'season': 2020,
  'conference': 'Southern',
  'position': 'WR',
  'pass_attempts': 1,
  'pass_completions': 1,
  'interceptions': 0,
  'completion_pct': 1.0,
  'pass_tds': 0,
  'pass_yds': 38,
  'yards_per_attempt': 38.0,
  'rush_attempts': 3,
  'longest_rush_attempt': 41,
  'rush_tds': 0,
  'rush_yds': 52,
  'yards_per_carry': 17.3,
  'longest_reception': 50,
  'receptions': 3,
  'receiving_tds': 0,
  'receiving_yds': 77,
  'yards_per_reception': 25.7},
 ('4034860', 'TCU'): {'cfbd_player_id': '4034860',
  'player_name': 'JD Spielman',
  'team': 'TCU',
  'season': 2020,
  'conference': 'Big 12',
  'position': 'WR',
  'pass_attempts': 1,
  'pass_completions': 1,
  'interceptions': 0,
  'completion_pct': 1.0,
  'pass_tds': 0,
  'pass_yds': 0,
  'yards_per_attempt': 0.0,
  'rush_attempts': 4,
  'longest_rush_attempt': 19,
  'rush_tds': 0,
  'rush_yds': 28,
  'yards_per_ca

### Combine Data

In [294]:
ids = nfl.import_ids()
player_ids = (
    ids[['gsis_id', 'pfr_id']][~ids['gsis_id'].isna()]
    .rename(columns={'gsis_id': 'player_id', 'pfr_id': 'pfr_player_id'})
    # .drop_duplicates(subset=['player_id'])
    .drop_duplicates(subset=["pfr_player_id"])
)
player_ids = player_ids[~player_ids["pfr_player_id"].isna()]




In [295]:
player_ids[player_ids["pfr_player_id"] == "CartKy01"]

,player_id,pfr_player_id
3904,00-0032606,CartKy01


In [296]:
raw_combine_data = nfl.import_combine_data(
    years=list(range(2010, 2025)),
    positions=list(COLLEGE_POSITIONS)
).rename(columns={'pfr_id': 'pfr_player_id'}).drop_duplicates(subset=['pfr_player_id', 'cfb_id'])
raw_combine_data.sort_values(["draft_year", "draft_ovr"], ascending=[False, True])#.sort_values(["draft_ovr"])
raw_combine_data

,season,draft_year,draft_team,draft_round,draft_ovr,pfr_player_id,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
3258,2010,NaN,None,NaN,NaN,AjirSe00,seyi-ajirotutu-1,Seyi Ajirotutu,WR,Fresno State,6-3,204.0,4.60,14.0,36.0,115.0,7.22,4.39
3261,2010,NaN,None,NaN,NaN,AlexDa00,danario-alexander-1,Danario Alexander,WR,Missouri,6-5,215.0,4.62,NaN,NaN,NaN,NaN,NaN
3268,2010,NaN,None,NaN,NaN,None,alric-arnett-1,Alric Arnett,WR,West Virginia,6-2,188.0,4.52,NaN,40.0,122.0,7.03,4.43
3272,2010,NaN,None,NaN,NaN,BankBr00,brandon-banks-1,Brandon Banks,WR,Kansas State,5-7,149.0,4.37,NaN,31.0,113.0,6.88,4.29
3276,2010,NaN,None,NaN,NaN,None,None,Chris Bell,WR,Norfolk State,6-2,211.0,4.50,15.0,35.0,117.0,6.76,4.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8312,2024,NaN,None,NaN,NaN,WillMi09,miyan-williams-1,Miyan Williams,RB,Ohio St.,5-9,229.0,NaN,NaN,NaN,NaN,NaN,NaN
8313,2024,2024.0,Philadelphia Eagles,6.0,185.0,WilsJo03,johnny-wilson-3,Johnny Wilson,WR,Florida St.,6-6,231.0,4.52,NaN,37.0,128.0,NaN,4.11
8315,2024,2024.0,Pittsburgh Steelers,3.0,84.0,WilsRo02,roman-wilson-1,Roman Wilson,WR,Michigan,5-11,185.0,4.39,12.0,NaN,NaN,NaN,NaN
8317,2024,2024.0,Kansas City Chiefs,1.0,28.0,WortXa00,xavier-worthy-1,Xavier Worthy,WR,Texas,5-11,165.0,4.21,NaN,41.0,131.0,NaN,NaN


In [308]:
raw_combine_data.size
raw_combine_data[~raw_combine_data["draft_year"].isna()].size

18432

In [303]:
player_ids[player_ids["pfr_player_id"] == "SmitKe03"]

,player_id,pfr_player_id
5024,00-0030968,SmitKe03


In [301]:
raw_combine_data[raw_combine_data["pfr_player_id"] == "SmitKe03"]

,season,draft_year,draft_team,draft_round,draft_ovr,pfr_player_id,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
3856,2011,NaN,None,NaN,NaN,SmitKe03,keith-smith-2,Keith Smith,WR,San Jose State,6-2,214.0,4.65,NaN,NaN,NaN,NaN,NaN


In [297]:
raw = player_ids.merge(raw_combine_data, on=['pfr_player_id'], how='inner')
raw

,player_id,pfr_player_id,season,draft_year,draft_team,draft_round,draft_ovr,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
0,00-0039918,WillCa03,2024,2024.0,Chicago Bears,1.0,1.0,caleb-williams-3,Caleb Williams,QB,USC,6-1,214.0,NaN,NaN,NaN,NaN,NaN,NaN
1,00-0039851,MayeDr00,2024,2024.0,New England Patriots,1.0,3.0,drake-maye-1,Drake Maye,QB,North Carolina,6-4,223.0,NaN,NaN,NaN,NaN,NaN,NaN
2,00-0039910,DaniJa02,2024,2024.0,Washington Commanders,1.0,2.0,jayden-daniels-1,Jayden Daniels,QB,LSU,6-4,210.0,NaN,NaN,NaN,NaN,NaN,NaN
3,00-0039732,NixxBo00,2024,2024.0,Denver Broncos,1.0,12.0,bo-nix-1,Bo Nix,QB,Oregon,6-2,214.0,NaN,NaN,NaN,NaN,NaN,NaN
4,00-0039917,PeniMi00,2024,2024.0,Atlanta Falcons,1.0,8.0,michael-penix-jr-1,Michael Penix Jr.,QB,Washington,6-2,216.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1353,00-0027606,HollTr00,2010,2010.0,Houston Texans,6.0,197.0,trindon-holliday-1,Trindon Holliday,WR,LSU,5-5,166.0,4.34,10.0,42.0,116.0,6.54,4.48
1354,00-0027795,GettDa00,2010,2010.0,Carolina Panthers,6.0,198.0,david-gettis-1,David Gettis,WR,Baylor,6-3,217.0,4.43,15.0,34.5,124.0,6.94,4.41
1355,00-0027608,WillKy01,2010,2010.0,San Francisco 49ers,6.0,206.0,kyle-williams-8,Kyle Williams,WR,Arizona State,5-10,188.0,4.40,11.0,33.0,118.0,7.00,4.19
1356,00-0027804,BrowLe00,2010,2010.0,Buffalo Bills,7.0,209.0,levi-brown-1,Levi Brown,QB,Troy,6-3,229.0,4.93,20.0,31.5,106.0,7.07,4.43


In [298]:
raw[raw["player_id"] == "00-0031118"]

,player_id,pfr_player_id,season,draft_year,draft_team,draft_round,draft_ovr,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
1000,00-0031118,BrowPh00,2014,NaN,None,NaN,NaN,corey-brown-2,Corey Brown,WR,Ohio State,5-11,178.0,4.51,NaN,33.0,116.0,7.16,4.22


In [300]:
# raw_combine_data.groupby('pfr_player_id').size().sort_values(ascending=False)
raw.groupby('pfr_player_id').size().sort_values(ascending=False)

pfr_player_id
AbanIs00    1
PageEr00    1
PatmDe00    1
PascZa00    1
ParkPr00    1
           ..
GreeAl00    1
GreeA.00    1
GrayNo00    1
GrayMa00    1
ZappBa00    1
Length: 1358, dtype: int64

### Draft History

In [243]:
raw_draft_data = nfl.import_draft_picks(list(range(2020, 2020 + 1))).rename(columns={'gsis_id': 'player_id'}).drop_duplicates(subset=['player_id', 'season'])
raw_draft_data

,season,round,pick,team,player_id,pfr_player_id,cfb_player_id,pfr_player_name,hof,position,category,side,college,age,to,allpro,probowls,seasons_started,w_av,car_av,dr_av,games,pass_completions,pass_attempts,pass_yards,pass_tds,pass_ints,rush_atts,rush_yards,rush_tds,receptions,rec_yards,rec_tds,def_solo_tackles,def_ints,def_sacks
11121,2020,1,1,CIN,00-0036442,BurrJo01,joe-burrow-1,Joe Burrow,False,QB,QB,O,LSU,23.0,2025.0,0,3,5,63.0,None,63.0,77.0,1921.0,2806.0,20810.0,157.0,51.0,239.0,847.0,12.0,0.0,0.0,0.0,1.0,NaN,NaN
11122,2020,1,2,WAS,00-0036321,YounCh04,chase-young-1,Chase Young,False,DE,DL,D,Ohio St.,21.0,2025.0,0,1,2,29.0,None,20.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,110.0,NaN,32.0
11123,2020,1,3,DET,00-0036274,OkudJe00,jeffrey-okudah-1,Jeff Okudah,False,CB,DB,D,Ohio St.,21.0,2025.0,0,0,2,11.0,None,7.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,157.0,2.0,NaN
11124,2020,1,4,NYG,00-0036386,ThomAn02,andrew-thomas-2,Andrew Thomas,False,T,OL,O,Georgia,21.0,2025.0,0,0,6,33.0,None,33.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,1.0,7.0,NaN,NaN
11125,2020,1,5,MIA,00-0036212,TagoTu00,tua-tagovailoa-1,Tua Tagovailoa,False,QB,QB,O,Alabama,22.0,2025.0,0,1,6,52.0,None,52.0,78.0,1647.0,2421.0,18166.0,120.0,59.0,173.0,473.0,6.0,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11371,2020,7,251,SEA,00-0036438,SullSt00,stephen-sullivan-1,Stephen Sullivan,False,TE,TE,O,LSU,23.0,2024.0,0,0,0,1.0,None,0.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.0,171.0,0.0,3.0,NaN,NaN
11372,2020,7,252,DEN,00-0036439,ClevTy00,tyrie-cleveland-1,Tyrie Cleveland,False,WR,WR,O,Florida,22.0,2022.0,0,0,0,0.0,None,0.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,91.0,0.0,6.0,NaN,NaN
11373,2020,7,253,MIN,00-0036354,HintKy00,None,Kyle Hinton,False,G,OL,O,Washburn,22.0,2025.0,0,0,0,3.0,None,0.0,52.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
11374,2020,7,254,DEN,00-0036440,TuszDe00,None,Derrek Tuszka,False,LB,LB,D,North Dakota St.,23.0,2022.0,0,0,0,3.0,None,1.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.0,NaN,2.0


In [180]:
raw = raw_draft_data[raw_draft_data["position"].isin(COLLEGE_POSITIONS)].copy()
raw = player_ids.merge(raw, on=['player_id', "pfr_player_id"], how='inner')

In [181]:
raw #.groupby('gsis_id').size().sort_values(ascending=False)

,player_id,pfr_player_id,season,round,pick,team,cfb_player_id,pfr_player_name,hof,position,category,side,college,age,to,allpro,probowls,seasons_started,w_av,car_av,dr_av,games,pass_completions,pass_attempts,pass_yards,pass_tds,pass_ints,rush_atts,rush_yards,rush_tds,receptions,rec_yards,rec_tds,def_solo_tackles,def_ints,def_sacks
0,00-0038400,McKeTa01,2023,6,188,PHI,tanner-mckee-1,Tanner McKee,False,QB,QB,O,Stanford,23.0,2025.0,0,0,0,2.0,None,2.0,6.0,54.0,88.0,597.0,5.0,1.0,13.0,7.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,00-0039150,YounBr01,2023,1,1,CAR,bryce-young-1,Bryce Young,False,QB,QB,O,Alabama,22.0,2025.0,0,0,2,23.0,None,23.0,46.0,853.0,1389.0,8291.0,49.0,30.0,136.0,718.0,8.0,0.0,0.0,0.0,NaN,NaN,NaN
2,00-0039152,LeviWi00,2023,2,33,TEN,will-levis-1,Will Levis,False,QB,QB,O,Kentucky,24.0,2024.0,0,0,2,11.0,None,11.0,21.0,339.0,556.0,3899.0,21.0,16.0,70.0,240.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN
3,00-0039163,StroCJ00,2023,1,2,HOU,cj-stroud-1,C.J. Stroud,False,QB,QB,O,Ohio St.,21.0,2025.0,0,1,3,33.0,None,33.0,46.0,928.0,1454.0,10876.0,62.0,25.0,139.0,609.0,4.0,1.0,0.0,0.0,NaN,NaN,NaN
4,00-0038550,HookHe00,2023,3,68,DET,hendon-hooker-1,Hendon Hooker,False,QB,QB,O,Tennessee,25.0,2024.0,0,0,0,0.0,None,0.0,3.0,6.0,9.0,62.0,0.0,0.0,5.0,2.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1508,00-0019709,RedmCh00,2000,3,75,BAL,chris-redman-1,Chris Redman,False,QB,QB,O,Louisville,23.0,2011.0,0,0,0,9.0,None,3.0,30.0,286.0,500.0,3179.0,21.0,14.0,33.0,25.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1509,00-0019484,MorrSa00,2000,5,156,BUF,sammy-morris-1,Sammy Morris,False,RB,RB,O,Texas Tech,23.0,2011.0,0,0,3,27.0,None,5.0,144.0,0.0,0.0,0.0,0.0,0.0,736.0,3053.0,26.0,166.0,1258.0,1.0,54.0,NaN,NaN
1510,00-0019599,BulgMa00,2000,6,168,NOR,marc-bulger-1,Marc Bulger,False,QB,QB,O,West Virginia,23.0,2009.0,0,2,8,57.0,None,NaN,96.0,1969.0,3171.0,22814.0,122.0,93.0,118.0,300.0,8.0,4.0,21.0,0.0,NaN,NaN,NaN
1511,00-0019596,BradTo00,2000,6,199,NWE,tom-brady-1,Tom Brady,False,QB,QB,O,Michigan,23.0,2022.0,3,15,21,184.0,None,168.0,335.0,7753.0,12050.0,89214.0,649.0,212.0,693.0,1123.0,28.0,3.0,65.0,0.0,1.0,NaN,NaN


In [189]:
def _safe_float(val) -> Optional[float]:
    try:
        if pd.isna(val):
            return None
        return float(val)
    except (TypeError, ValueError):
        return None

def _safe_int(val) -> Optional[int]:
    try:
        if pd.isna(val):
            return None
        return int(float(val))
    except (TypeError, ValueError):
        return None

with get_session(engine) as session:
    for _, row in raw_draft_data.iterrows():
        player_id = str(row.get("player_id") or "")
        if not player_id:
            continue
        # print(_safe_int(row.get("round")))
        # print(_safe_int(row.get("pick")))
        # print(_safe_int(row.get("year")))
        # print(row.get("team"))
        # print(row.get("college"))